**About the environment:** this notebook is written for **SageMath** and uses the module `hilbert_kunz.py`, which must be in the same folder.

# The Chiang–Hung algorithm: Hilbert–Kunz functions of $\sum_i\prod_j X_{i,j}^{d_{i,j}}$

This notebook runs the algorithm of Chiang and Hung [CH98], which builds on the techniques of Han and Monsky [HM93] (notebook 02) and extends them from diagonal hypersurfaces to
$$f=\sum_{i=1}^m\ \prod_{j=1}^{t_i}X_{i,j}^{d_{i,j}}$$
[Notes, §1]. We write $R=k[[X_{i,j}]]/(f)$, $k=\mathbb F_p$, and $\operatorname{HK}_e(R,I)=\ell(R/I^{[p^e]})$ as in the notes; [CH98] writes $e_n(f)$ for $\operatorname{HK}_n(R)$.

By [CH98, Thm 1.1] the Hilbert–Kunz function has the form $\operatorname{HK}_e(R)=c\,p^{a e}+\sum\Delta(e)$, where $a=N-1=\dim R$ ($N$ is the number of variables), $c=e_{\mathrm{HK}}(R)\in\mathbb Q$, and the $\Delta$'s come from the eigenvalues of an integer matrix $\tilde B$. All the computations take place in the Han–Monsky ring $\Gamma_k$ (notebook 01); the invariant $\alpha=\dim_k(-/T-)$ of the notes is called $D_F$ in [CH98].

Page numbers refer to the journal pages of [CH98]. Run the next cell, then any of the examples; how the algorithm works is explained afterwards (§5).


Main functions (the polynomial is given directly, over `GF(p)`):
* `HK_function_disjoint_monomials(f)`: the Hilbert–Kunz function, one formula per residue class of $e$. Here $f$ is a sum of monomials in pairwise disjoint sets of variables, such as $(XY)^4+T^4+U^4+W^4$, and every variable of the ring must appear in $f$. The result `H` can be evaluated, `H(e)`, by the recursion of [CH98, p. 516];
* `FS_function_disjoint_monomials(f)`: the F-signature function (§4);
* `HK_groebner(f, e)`: an independent check with a Gröbner basis;
* both functions take an optional `ideal=[a_1, ..., a_N]` (one exponent per variable, in the order of the ring) to replace the maximal ideal by $I=(X_1^{a_1},\dots,X_N^{a_N})$ (§3).

Lower-level functions take the **exponent lists** of $f$, one list per monomial: $(XY)^4+T^4+U^4+W^4$ is `[[4,4],[4],[4],[4]]`. They are
* `eHK(p, exps)`: the Hilbert–Kunz multiplicity $c$ (exact, fast);
* `hk(p, exps, e)`: the exact value at $e$, via the recursion of [CH98, p. 516]; `brute(p, exps, e)`: the same with a Gröbner basis;
* `F_signature(p, exps)`, `fs(p, exps, e)`, `fs_brute(p, exps, e)`: the F-signature, and values of the F-signature function;
* `ChiangHung(p, t, d)`: the recursion itself, for $f=\sum_i(X_{i,1}\cdots X_{i,t_i})^{d_i}$, with its matrices (§5).

In [1]:
load("hilbert_kunz.py")

## 1. The examples of [CH98]

### 1.0 Warm-up: a diagonal hypersurface (Han–Monsky)

$f=X^4+T^4+U^4+W^4$ over $\mathbb F_5$, quoted in [CH98, p. 525] from [HM93]; this is [Notes, Example 1.3]:
$$e\longmapsto\frac{168}{61}\,5^{3e}-\frac{107}{61}\,3^e.$$

In [2]:
S.<X,T,U,W> = GF(5)[]
HK_function_disjoint_monomials(X^4 + T^4 + U^4 + W^4)

HK_e = 168/61 * 5^(3e) - 107/61 * 3^e   for e >= 0

### 1.1 Example 1 [p. 523]

$f=(XY)^4+T^4+U^4+W^4$ over $\mathbb F_5$:
$$e\longmapsto\frac{285166}{96099}\,5^{4e}+\frac{145}{311}\,3^{e}-\frac{752}{309}\,7^{e}.$$

In [3]:
S.<X,Y,T,U,W> = GF(5)[]
f = (X*Y)^4 + T^4 + U^4 + W^4
H = HK_function_disjoint_monomials(f)
print(H)
print([H(e) for e in range(4)], HK_groebner(f, 1), HK_groebner(f, 2))

HK_e = 285166/96099 * 5^(4e) - 752/309 * 7^e + 145/311 * 3^e   for e >= 0
[1, 1839, 1159033, 724466711] 1839 1159033


### 1.2 Example 2 [pp. 523–525]

$f=(XYZ)^4+T^4+U^4+W^4$ over $\mathbb F_5$:
$$e\longmapsto\frac{378803300}{125571523}\,5^{5e}\;{\color{red}-}\;\frac{5837}{1561}\,3^{e}+\frac{53520}{13277}-\frac{4042}{1751}\,35^{e}.$$

**Erratum.** The paper prints $+\frac{5837}{1561}3^e$. With that sign the value at $e=1$ is $14613201/1561$, which is not an integer, whereas $e_1=9339$. $\tilde A(\mu)$ is $6\times6$ (printed on p. 524).

In [4]:
S.<X,Y,Z,T,U,W> = GF(5)[]
f = (X*Y*Z)^4 + T^4 + U^4 + W^4
print(HK_function_disjoint_monomials(f))
print(matrix(ZZ, ChiangHung(5, [3,1,1,1], [4,4,4,4]).At(1)))
print("Groebner basis, e = 1:", HK_groebner(f, 1))

HK_e = 378803300/125571523 * 5^(5e) - 4042/1751 * 35^e
       - 5837/1561 * 3^e + 53520/13277
       for e >= 0
[  1   0   0   0   0   0]
[  0   3 -64   0 -48   0]
[  0   0   4  -3  18 -12]
[  0   0  -3   4 -24  30]
[  0   0   0   0  20 -15]
[  0   0   0   0 -15  20]
Groebner basis, e = 1: 9339


### 1.3 Unequal exponents: $X^{d_1}Y^{d_2}Z^4+T^4+U^4+W^4$ [p. 525]

For $d_1,d_2\le4$, Eq. (21) writes the function as a combination of Example 2, Example 1 and §1.0:
$$\frac{d_1d_2}{16}[\text{Ex. 2}]+\Big[\Big(1-\frac{d_1}4\Big)\frac{d_2}4+\Big(1-\frac{d_2}4\Big)\frac{d_1}4\Big]5^{e}[\text{Ex. 1}]+\Big(1-\frac{d_1}4\Big)\Big(1-\frac{d_2}4\Big)5^{2e}[\text{H–M}]$$
(with the sign correction in Example 2). For $d_1=2$, $d_2=3$:
$$\frac{42296881166719}{14293304176998}\,5^{5e}-\frac{107}{488}\,75^{e}-\frac{43757}{21012}\,35^{e}+\frac{145}{622}\,15^{e}-\frac{17511}{12488}\,3^{e}+\frac{20070}{13277}.$$

In [5]:
S.<X,Y,Z,T,U,W> = GF(5)[]
HK_function_disjoint_monomials(X^2*Y^3*Z^4 + T^4 + U^4 + W^4)

HK_e = 42296881166719/14293304176998 * 5^(5e) - 107/488 * 75^e
       - 43757/21012 * 35^e + 145/622 * 15^e - 17511/12488 * 3^e
       + 20070/13277
       for e >= 0

In [6]:
E2 = lambda n: 378803300/125571523*5^(5*n) - 5837/1561*3^n + 53520/13277 - 4042/1751*35^n   # sign corrected
E1 = lambda n: 285166/96099*5^(4*n) + 145/311*3^n - 752/309*7^n
HM = lambda n: 168/61*5^(3*n) - 107/61*3^n
paper = lambda d1, d2, n: (d1*d2/16*E2(n) + ((1-d1/4)*d2/4 + (1-d2/4)*d1/4)*5^n*E1(n)
                           + (1-d1/4)*(1-d2/4)*5^(2*n)*HM(n))
print("matches the paper for all d1, d2 <= 4, e < 4:",
      all(hk(5, [[d1,d2,4],[4],[4],[4]], n) == paper(d1, d2, n)
          for d1 in range(1,5) for d2 in range(1,5) for n in range(4)))
f = X^2*Y^3*Z^4 + T^4 + U^4 + W^4
print("Groebner basis e = 1, 2 for d1=2, d2=3:", HK_groebner(f, 1), HK_groebner(f, 2))

matches the paper for all d1, d2 <= 4, e < 4: True
Groebner basis e = 1, 2 for d1=2, d2=3: 9159 28894787


### 1.4 Example 3 [pp. 525–526]: a non-diagonalisable $\tilde B$ ([Notes, Example 1.5])

$f=(XYZT)^6+U^6+W^6+S^6$ over $\mathbb F_7$:
$$e\longmapsto\frac{2050747718345259}{429658164277013}\,7^{6e}+\frac{104440051}{294115}\,3^{e}-\frac{267597}{74480}\,7^{3e}-\frac{312186671451}{878707312}\,7^{e}+\frac{164565}{2801}\,e\,7^{e}.$$
The Jordan form of $\tilde B=\tilde A(1)$ (p. 526) has $2\times2$ blocks for the eigenvalues $7$ and $49$. The block for $7$ produces the term $e\,7^e$, whose coefficient is **linear in $e$**. The block for $49$ gets coefficient $0$.

In [7]:
S.<X,Y,Z,T,U,W,V> = GF(7)[]
HK_function_disjoint_monomials((X*Y*Z*T)^6 + U^6 + W^6 + V^6)

HK_e = 2050747718345259/429658164277013 * 7^(6e)
       - 267597/74480 * 7^(3e)
       + (164565e/2801 - 312186671451/878707312) * 7^e
       + 104440051/294115 * 3^e
       for e >= 0

In [8]:
B = matrix(QQ, ChiangHung(7, [4,1,1,1], [6,6,6,6]).At(1))
print(B); print()
print(B.jordan_form())

[    0     0     0     0     0     0     0     0]
[    0     3 -1296     0  -864     0  -216     0]
[    0     0     4    -3    24   -16    72   -48]
[    0     0    -3     4   -56    72  -408   504]
[    0     0     0     0    28   -21   126   -84]
[    0     0     0     0   -21    28  -294   378]
[    0     0     0     0     0     0   196  -147]
[    0     0     0     0     0     0  -147   196]

[343|  0|  0|  0|  0   0|  0   0]
[---+---+---+---+-------+-------]
[  0|  3|  0|  0|  0   0|  0   0]
[---+---+---+---+-------+-------]
[  0|  0|  1|  0|  0   0|  0   0]
[---+---+---+---+-------+-------]
[  0|  0|  0|  0|  0   0|  0   0]
[---+---+---+---+-------+-------]
[  0|  0|  0|  0| 49   1|  0   0]
[  0|  0|  0|  0|  0  49|  0   0]
[---+---+---+---+-------+-------]
[  0|  0|  0|  0|  0   0|  7   1]
[  0|  0|  0|  0|  0   0|  0   7]


**Remark on Monsky's example (p. 526).** The paper reports a $p=19$ example printed as $XYZT+U^{19}+V^{19}+W^{19}$. As printed, it gives a single term $c\,19^{6e}$ (below), and the same happens for $p=3,5$. So the printed hypersurface is probably missing exponents. The paper's multiplicity $\frac{19017043668514596550}{70272691919506647}$ does not occur for $(XYZT)^a+U^b+V^b+W^b$ with $a,b\le 38$ (checked with `eHK`), so the intended hypersurface remains unclear.

In [9]:
for p in [3, 5, 19]:
    S.<X,Y,Z,T,U,V,W> = GF(p)[]
    print(HK_function_disjoint_monomials(X*Y*Z*T + U^p + V^p + W^p))

HK_e = 65/27 * 3^(6e)   for e >= 1
HK_e = 369/125 * 5^(6e)   for e >= 1
HK_e = 25345/6859 * 19^(6e)   for e >= 1


### 1.5 Kunz's example [Kun76, Example 4.3], [Notes, Examples 1.2(1)]

$R=\mathbb F_5[[t,x,y,z]]/(t^3-x^2y^2z^2)$:
$$\operatorname{HK}_e(R)=\frac{26}{9}\,5^{3e}-\frac83\,5^{e}+(-1)^e\,\frac{16}{9}\qquad(e\ge1),$$
a quasi-polynomial in $5^e$. Nonzero coefficients are allowed: over $\overline{\mathbb F}_p$ they can be absorbed by rescaling one variable in each monomial, and lengths do not change under field extension.

In [10]:
S.<t,x,y,z> = GF(5)[]
H = HK_function_disjoint_monomials(t^3 - x^2*y^2*z^2)
print(H)
print([H(e) for e in range(1, 5)] == [26/9*5^(3*e) - 8/3*5^e + (-1)^e*16/9 for e in range(1, 5)])

HK_e = 26/9 * 5^(3e) - 8/3 * 5^e + 16/9   for even e >= 2
HK_e = 26/9 * 5^(3e) - 8/3 * 5^e - 16/9   for odd e >= 1
True


[Notes, Examples 1.2(2)] (an observation of I. Smirnov): $\operatorname{HK}_e(R[[t]])=p^e\operatorname{HK}_e(R)$. A quick check with Gröbner bases, adjoining a variable to $\mathbb F_p[[x,y]]/(x^3+y^4)$:

In [11]:
for p, e in [(3, 1), (3, 2), (5, 1), (5, 2)]:
    A.<x,y> = GF(p)[]; B.<x,y,t> = GF(p)[]
    hkA = A.ideal([x^3 + y^4] + [g^(p^e) for g in A.gens()]).vector_space_dimension()
    hkB = B.ideal([x^3 + y^4] + [g^(p^e) for g in B.gens()]).vector_space_dimension()
    print(p, e, hkA, hkB, hkB == p^e * hkA)

3 1 9 27 True
3 2 27 243 True
5 1 14 70 True
5 2 75 1875 True


## 2. New examples: linear coefficients in fewer variables

### 2.1 Six variables, $p=5$

$f=(XYZ)^6+U^4+V^4+W^4$ over $\mathbb F_5$:
$$e\longmapsto\frac{381550975}{125651966}\,5^{5e}-\frac{74448}{29767}\,35^{e}+\Big(\frac{163746985}{352557458}-\frac{361260}{13277}\,e\Big)\qquad(e\ge0).$$
The linear coefficient sits on the base $1=p^0$, so the function contains a bare term $-\frac{361260}{13277}\,e$.

This is the smallest example found in a search over $\sum_i(X_{i,1}\cdots X_{i,t_i})^{d_i}$ with $d_i\le 8,9,10,12$ for $p=2,3,5,7$. There were **no** linear coefficients for:
* $\le5$ variables and $p\le7$;
* 6 variables and $p\in\{2,3\}$;
* 7 variables and $p=2$.

In [12]:
S.<X,Y,Z,U,V,W> = GF(5)[]
f = (X*Y*Z)^6 + U^4 + V^4 + W^4
H = HK_function_disjoint_monomials(f)
print(H)
print([H(e) for e in range(5)])
print("Groebner basis, e = 1, 2:", HK_groebner(f, 1), HK_groebner(f, 2))

HK_e = 381550975/125651966 * 5^(5e) - 74448/29767 * 35^e
       + (163746985/352557458 - 361260e/13277)
       for e >= 0
[1, 9375, 29650885, 92668651155, 289589866458025]
Groebner basis, e = 1, 2: 9375 29650885


### 2.2 The family $(XYZ)^{p+1}+U^{p-1}+V^{p-1}+W^{p-1}$

| $p$ | linear term |
|---|---|
| 3 | none |
| 5 | $-\frac{361260}{13277}\,e$ |
| 7 | $-\frac{1365280}{8403}\,e$ |
| 11 | $-\frac{1414380528}{1529975}\,e$ |

It appears to hold for every $p\ge5$. This is an observation, not a theorem.

In [13]:
for p in [3, 5, 7, 11]:
    S.<X,Y,Z,U,V,W> = GF(p)[]
    print("p =", p)
    print(HK_function_disjoint_monomials((X*Y*Z)^(p+1) + U^(p-1) + V^(p-1) + W^(p-1)))
    print()

p = 3
HK_e = 311/209 * 3^(5e) - 96/133 * 15^e + 18/77   for e >= 0

p = 5
HK_e = 381550975/125651966 * 5^(5e) - 74448/29767 * 35^e
       + (163746985/352557458 - 361260e/13277)
       for e >= 0

p = 7
HK_e = 19218491885/4024793313 * 7^(5e) - 1802/513 * 7^(2e)
       + (-1365280e/8403 - 18525010/70610409)
       for e >= 0

p = 11
HK_e = 14958246523701/1757238694375 * 11^(5e) - 505840/97831 * 77^e
       + (-1414380528e/1529975 - 5481763262866/2340823500625)
       for e >= 0



## 3. Other ideals $(X^{a})$

Every function takes an optional `ideal`, which replaces the maximal ideal by $I=(X_1^{a_1},\dots,X_N^{a_N})$, one exponent per variable in the order of the ring. The function computed is then
$$e\longmapsto\operatorname{HK}_e(R,I)=\dim_{\mathbb F_p}\mathbb F_p[X]\big/\big(X_1^{a_1p^e},\dots,X_N^{a_Np^e},\,f\big).$$

| hypersurface and ideal | ring | `ideal` |
|---|---|---|
| $x^3+y^3+z^3+t^3$, $\ (x,y,z,t^2)$ | `S.<x,y,z,t>` | `[1,1,1,2]` |
| $(XY)^4+T^4+U^4+W^4$, $\ (X,Y^2,T,U^3,W)$ | `S.<X,Y,T,U,W>` | `[1,2,1,3,1]` |

(The lower-level functions take the ideal in the shape of `exps`, e.g. `[[1,2],[1],[3],[1]]`.)

For a diagonal hypersurface this agrees with the Han–Monsky code (notebook 02):

In [14]:
S.<x,y,z,t> = GF(7)[]
f = x^3 + y^3 + z^3 + t^3
print(HK_function_diagonal(f, ideal=[1,1,1,2]))
print(HK_function_disjoint_monomials(f, ideal=[1,1,1,2]))

HK_e = 489/170 * 7^(3e) - 149/170 * 3^e   for e >= 0
HK_e = 489/170 * 7^(3e) - 149/170 * 3^e   for e >= 0


In [15]:
S.<X,Y,T,U,W> = GF(5)[]
f = (X*Y)^4 + T^4 + U^4 + W^4
H = HK_function_disjoint_monomials(f, ideal=[1,2,1,3,1])
print(H)
print("e_HK(I) =", eHK(5, [[4,4],[4],[4],[4]], [[1,2],[1],[3],[1]]))
print([H(e) for e in range(4)], " Groebner basis, e = 1:", HK_groebner(f, 1, ideal=[1,2,1,3,1]))

HK_e = 8 * 5^(4e) - 5^e - 1   for e >= 0
e_HK(I) = 8
[6, 4994, 3124974, 1953124874]  Groebner basis, e = 1: 4994


The linear-coefficient example of §2.1, now with the ideal $(X,Y,Z,U,V,W^2)$. The term linear in $e$ disappears:

In [16]:
S.<X,Y,Z,U,V,W> = GF(5)[]
HK_function_disjoint_monomials((X*Y*Z)^6 + U^4 + V^4 + W^4, ideal=[1,1,1,1,1,2])

HK_e = 239284/60137 * 5^(5e) - 3510/847 * 45^e + 18600/8591   for e >= 0

## 4. F-signature functions

For a hypersurface with a pure-power monomial $Y^d$:

* `FS_function_disjoint_monomials(f)` gives the F-signature function $e\mapsto\operatorname{FS}_e(R)$, in the same layout as `HK_function_disjoint_monomials`;
* `fs(p, exps, e)` gives one value, and `F_signature(p, exps)` gives the F-signature $s(R)=\lim_e \operatorname{FS}_e(R)/q^{\dim R}$;
* `fs_brute(p, exps, e)` gives an independent check via Gröbner bases.

How it is computed is explained at the end of this section.

**Quotient singularities.** Their F-signature is $1/|G|$ when $p\nmid|G|$ [WY04, HL02]; compare notebook 02, §1.6. $A_n$ can be written as $XY+Z^{n+1}$, which is not diagonal:

In [17]:
p = 7
for n in range(1, 7):
    print("A_%d = XY + Z^%d:  s = %s" % (n, n+1, F_signature(p, [[1,1],[n+1]])))
print("E_6 = X^2+Y^3+Z^4:  s =", F_signature(p, [[2],[3],[4]]))
print("E_8 = X^2+Y^3+Z^5:  s =", F_signature(p, [[2],[3],[5]]), "   (p = 5:", F_signature(5, [[2],[3],[5]]), ")")
S.<X,Y,Z> = GF(p)[]
FS_function_disjoint_monomials(X*Y + Z^4)

A_1 = XY + Z^2:  s = 1/2
A_2 = XY + Z^3:  s = 1/3
A_3 = XY + Z^4:  s = 1/4
A_4 = XY + Z^5:  s = 1/5
A_5 = XY + Z^6:  s = 1/6
A_6 = XY + Z^7:  s = 1/7
E_6 = X^2+Y^3+Z^4:  s = 1/24
E_8 = X^2+Y^3+Z^5:  s = 1/120    (p = 5: 0 )


FS_e = 1/4 * 7^(2e) + 3/4   for e >= 0

**The examples of §1.** For instance, Example 1 has $\operatorname{FS}_e=1$ for every $e$ (F-pure, but not strongly F-regular), and Example 3 is not F-pure:

In [18]:
S.<X,Y,T,U,W> = GF(5)[]
print(FS_function_disjoint_monomials((X*Y)^4 + T^4 + U^4 + W^4))
S.<X,Y,Z,T,U,W,V> = GF(7)[]
print(FS_function_disjoint_monomials((X*Y*Z*T)^6 + U^6 + W^6 + V^6))

FS_e = 1   for e >= 0
FS_e = 0   for e >= 1


**Unusual terms carry over.** A sign-alternating term $(-4)^e$, and a term $6^{\lfloor e/2\rfloor}$ coming from the eigenvalues $\pm\sqrt6$:

In [19]:
S.<X,Y,T,U,W> = GF(7)[]
print(FS_function_disjoint_monomials((X*Y)^2 + T^3 + U^3 + W^3))
print(FS_function_disjoint_monomials((X*Y)^3 + T^3 + U^3 + W^6))
print("check against Groebner bases:", [fs(7, [[2,2],[3],[3],[3]], e) for e in range(2)], [fs_brute(7, [[2,2],[3],[3],[3]], e) for e in range(2)])

FS_e = 131233/954785 * 7^(4e) + 7320/9131 * 19^e
       + 3368/55315 * (-4)^e
       for e >= 0
FS_e = 16581417/2294388410 * 7^(4e) + 40257/64874 * 13^e
       + 349771828/939661585 * 6^(e/2)
       for even e >= 0
FS_e = 16581417/2294388410 * 7^(4e) + 40257/64874 * 13^e
       - 393653042/939661585 * 6^((e - 1)/2)
       for odd e >= 1
check against Groebner bases: [1, 345] [1, 345]


**A strongly F-regular ring whose F-signature function has a term linear in $e$:** $XYZT+U^6+V^4+W^4$ over $\mathbb F_7$. Its F-signature is positive. The value at $e=1$ is checked against the colon formula; at $e=2$ the three possible choices of pure power $Y^d$ give the same value, although they use different ideals.

In [20]:
p, exps = 7, [[1,1,1,1],[6],[4],[4]]
S.<X,Y,Z,T,U,V,W> = GF(p)[]
print(FS_function_disjoint_monomials(X*Y*Z*T + U^6 + V^4 + W^4))
print("s(R) =", F_signature(p, exps))
print("FS_1:", fs(p, exps, 1), "  colon formula:", fs_brute(p, exps, 1))
print("FS_2 with Y = U, V, W:", [fs(p, exps, 2, var=v) for v in (1, 2, 3)])

FS_e = 11988461252/197214872337 * 7^(6e) + 565063/1608768 * 7^(3e)
       + (18599577623/31633463232 - 6415e/33612) * 7^e
       for e >= 0
s(R) = 11988461252/197214872337
FS_1: 7275   colon formula: 7275
FS_2 with Y = U, V, W: [841436981, 841436981, 841436981]


**How it is computed.** Huneke–Leuschke [HL02], [Notes, Proposition 4.1]: for $R$ Gorenstein, $I$ a parameter ideal and $u$ a socle element of $R/I$,
$$\operatorname{FS}_e(R)=\ell\big(R/I^{[q]}\big)-\ell\big(R/(I+(u))^{[q]}\big)=\ell(R/I)\,q^{\dim R}-\operatorname{HK}_e\big(R,I+(u)\big).$$

If $f$ contains a pure power $Y^d$, take $I$ = all the other variables, as in [Notes, Lemma 4.2] for diagonal hypersurfaces. Then:
* $R/I=k[Y]/(Y^d)$, so $\ell(R/I)=d$;
* $u=Y^{d-1}$;
* $I+(u)$ is of the form $(X_{i,j}^{a_{i,j}})$ treated in §3.

The pure power used is the one with the smallest $d$; `fs(..., var=i)` chooses another, with the same result. If no monomial is a pure power (e.g. $XY+ZW$), there is no monomial parameter ideal of this kind, and the functions say so.

Note also that in $f=\sum_i\prod_j X_{i,j}^{d_{i,j}}$ different monomials use different variables. So, for example, $D_n=X^2+Y^2Z+Z^{n-1}$ and $E_7=X^2+Y^3+YZ^3$ are outside the scope of this algorithm.

## 5. How the algorithm works, on Example 1 of [CH98]

We follow $f=(XY)^4+T^4+U^4+W^4$ over $\mathbb F_5$, so $t=(2,1,1,1)$, $d=(4,4,4,4)$ and $a=\sum t_i-1=4$.

**Step A: equal exponents** [CH98, Eq. (21), Corollary 1]. Here $e_n(f)=\operatorname{HK}_n(R)$ in the notation of [CH98]. If the exponents inside a monomial differ, $e_n(f)$ is a rational combination of $p^{kn}\,e_n(f_k)$ for hypersurfaces $f_k$ with equal exponents inside every monomial. `_reduce` performs this decomposition. The next cell shows it for $X^2Y^3Z^4+T^4+U^4+W^4$: each term is $(\text{numerator},k,t,d)$, and the common denominator is printed second.

In [21]:
_reduce([[2,3,4],[4],[4],[4]])

[(3/8, 0, [3, 1, 1, 1], [4, 4, 4, 4], [1, 1, 1, 1]),
 (1/2, 1, [2, 1, 1, 1], [4, 4, 4, 4], [1, 1, 1, 1]),
 (1/8, 2, [1, 1, 1, 1], [4, 4, 4, 4], [1, 1, 1, 1])]

**Step B: the state space.**

- The index set $J$ consists of tuples $\sigma$ with $\sigma_i\in\{0^*,0,1,\dots,t_i-2\}$. Here $0^*$ is printed as $-1$, and the order is the linear order of [CH98, p. 513].
- The recursion runs over the $p$-adic digits $b_i(\mu)$ of $1/d_i$ [p. 512].
- Everything at step $\mu$ depends only on the pairs $(b_i(\mu),\,p^\mu\bmod d_i)$.

In [22]:
X = ChiangHung(5, [2,1,1,1], [4,4,4,4])
print(X)
print("J =", X.J)
print("digits of 1/4 in base 5:", [X.b(0, mu) for mu in range(6)])
print("state(0), state(1), state(2):", X.state(0), X.state(1), X.state(2))

Chiang-Hung recursion for f = (X11X12)^4 + X21^4 + X31^4 + X41^4 over GF(5)
J = [(-1, -1, -1, -1), (0, -1, -1, -1)]
digits of 1/4 in base 5: [0, 1, 1, 1, 1, 1]
state(0), state(1), state(2): ((0, 1), (0, 1), (0, 1), (0, 1)) ((1, 1), (1, 1), (1, 1), (1, 1)) ((1, 1), (1, 1), (1, 1), (1, 1))


**Step C: the integer recursion** [CH98, p. 516]:
$$\tilde v(n,\mu)=\tilde A(\mu)\,\tilde v(n,\mu+1)+p^{a(n-\mu)}\,\tilde\Gamma(\mu),\qquad e_n=\text{second entry of }\tilde v(n,0).$$
The paper prints, for $\mu\ge1$: $\tilde A(\mu)$ as below, $\tilde\Gamma(\mu)=(0,1852,248,1000)$ and $\tilde v(1,1)=(1,1839,249,1001)$ [p. 523].

In [23]:
print(matrix(ZZ, X.At(1)))
print("Gamma~(1) =", X.Gt(1), "   v~(1,1) =", X.vbase(1))
print("A~(0) ="); print(matrix(ZZ, X.At(0)))

[  1   0   0   0]
[  0   3 -16   0]
[  0   0   4  -3]
[  0   0  -3   4]
Gamma~(1) = [0, 1852, 248, 1000]    v~(1,1) = [1, 1839, 249, 1001]
A~(0) =
[0 0 0 0]
[0 1 0 0]
[0 0 1 0]
[0 0 0 1]


In [24]:
# running the recursion by hand for n = 3 ...
n = 3
v = vector(ZZ, X.vbase(n))
for mu in range(n-1, -1, -1):
    v = matrix(ZZ, X.At(mu))*v + 5^(X.a*(n-mu))*vector(ZZ, X.Gt(mu))
print("by hand:", v[1], "   X.hk(3):", X.hk(3), "   Groebner basis:", brute(5, [[4,4],[4],[4],[4]], 3))

by hand: 724466711    X.hk(3): 724466711    Groebner basis: 724466711


**Step D: closed form** [CH98, p. 517]. The data is periodic in $\mu$ after $\rho$ steps with period $\omega$; here $\rho=0$, $\omega=1$. So $\tilde v(n,0)=C\,\tilde B^{\tau(n)}\Phi(n)+p^{an}\mathbf z$ with $\tilde B=\tilde A(1)$. The Jordan form of $\tilde B$ determines which exponentials appear.

In [25]:
print("rho, omega =", X.period())
B = matrix(QQ, X.At(1))
print("eigenvalues of B~:", B.eigenvalues())
S.<X,Y,T,U,W> = GF(5)[]
HK_function_disjoint_monomials((X*Y)^4 + T^4 + U^4 + W^4)

rho, omega = (0, 1)
eigenvalues of B~: [7, 3, 1, 1]


HK_e = 285166/96099 * 5^(4e) - 752/309 * 7^e + 145/311 * 3^e   for e >= 0

### 5.1 Other ideals

For a monomial $X_1^{d_1}\cdots X_t^{d_t}$ acting on $k[X]/(X_j^{a_jq})$, the $k$-object is a direct sum of chains of monomials. Its class in $\Gamma_k$ is
$$\psi=\sum_{L\ge1}\big(N(L-1)-2N(L)+N(L+1)\big)\,\delta_L,\qquad N(L)=\prod_j\,(a_jq-Ld_j)^+,$$
where $N(L)$ counts the monomials $m$ with $m$ and $T^Lm$ both nonzero. Since $\psi$ depends linearly on $N$, two things follow.

* **Scaling.** Replacing $(a_j,d_j)$ by $(c\,a_j,c\,d_j)$ multiplies $\psi$ by $c$. This brings all the $a_j$ of a monomial to a common value $A$.
* **Step A** [CH98, Eq. (21)] goes through with $q$ replaced by $Aq$:
$$(Aq-Ld_j)^+(Aq-LD)^+=\tfrac{d_j}{D}\big((Aq-LD)^+\big)^2+Aq\big(1-\tfrac{d_j}{D}\big)(Aq-LD)^+\qquad(d_j\le D).$$

After this, each monomial has equal exponents $d$ and an equal ideal power $A$. The recursion of [CH98] then runs as before, with two changes:
* the $p$-adic digits of $A/d$ replace those of $1/d$;
* $Ap^\mu\bmod d$ replaces $p^\mu\bmod d$.

The integer part $\lfloor A/d\rfloor$ may exceed $p-1$. Only the step $\mu=0$ sees it, and the products there are computed exactly. The results are checked against Gröbner bases above.

## References

Labels follow the alpha style of the notes where possible; the BibTeX key used in `han_monsky.tex` is given in brackets, so each entry can be matched with the notes' `.bib` file.

- **[Notes]** J. Castillo-Rey. *The Hilbert–Kunz function and F-signature function of the diagonal hypersurfaces*, notes (`han_monsky.tex`). Numbering of results as in the compiled notes.
- **[CH98]** L. Chiang and Y.-C. Hung. *On Hilbert–Kunz functions of some hypersurfaces*, J. Algebra **199** (1998), 499–527.  [`chiangHKSomeHypersurfaces1998`]
- **[CSTZ24]** A. Caminata, S. Shideler, K. Tucker, F. Zerman. *F-signature functions of diagonal hypersurfaces*, arXiv:2403.12863 (2024).  [`shidelerFsignatureFunctionsDiagonal2024`]
- **[Han91]** C. Han. *The Hilbert–Kunz function of a diagonal hypersurface*, Ph.D. thesis, Brandeis University, 1991.  [`han1992hilbert`]
- **[HL02]** C. Huneke and G. J. Leuschke. *Two theorems about maximal Cohen–Macaulay modules*, Math. Ann. **324** (2002), no. 2, 391–404.  [`hunekeTwoTheoremsMaximal2004`]
- **[HM93]** C. Han and P. Monsky. *Some surprising Hilbert–Kunz functions*, Math. Z. **214** (1993), no. 1, 119–135. DOI: [10.1007/BF02572395](https://doi.org/10.1007/BF02572395).  [`hanmonskySurprising1993`]
- **[Kun76]** E. Kunz. *On Noetherian rings of characteristic p*, Amer. J. Math. **98** (1976), 999–1013.  [`kunz1976`]
- **[Mon83]** P. Monsky. *The Hilbert–Kunz function*, Math. Ann. **263** (1983), 43–49.  [`monskyHilbertKunzFunction1983`]
- **[MT04]** P. Monsky and P. Teixeira. *p-Fractals and power series — I. Some 2-variable results*, J. Algebra **280** (2004), 505–536. DOI: [10.1016/j.jalgebra.2004.05.016](https://doi.org/10.1016/j.jalgebra.2004.05.016).  [`monsky2004p`]
- **[MT06]** P. Monsky and P. Teixeira. *p-Fractals and power series — II. Some applications to Hilbert–Kunz theory*, J. Algebra **304** (2006), 237–255 (arXiv:math/0506584).  [`monskyPFractals2006`]
- **[WY04]** K.-i. Watanabe and K.-i. Yoshida. *Minimal relative Hilbert–Kunz multiplicity*, Illinois J. Math. **48** (2004), 273–294.
- **[Yos09]** K.-i. Yoshida. *Small Hilbert–Kunz multiplicity and $(A_1)$-type singularity*, Proceedings of the 4th Japan-Vietnam Joint Seminar on Commutative Algebra, Meiji University (2009).  [`Yo09`]